# PyTorch Autograd — From Scratch

Autograd → manual gradients → matrix operations → manual losses → linear regression → logistic regression → interview questions.

`torch.no_grad()` for parameter updates. No built-in loss functions or optimizers.

## 1. Basic Autograd

Autograd calculates derivatives of a computation graph.

$$
 y = x^2, \qquad \frac{dy}{dx}=2x
$$

In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y

tensor(9., grad_fn=<PowBackward0>)

In [2]:
y.backward()
x.grad

tensor(6.)

## 2. Chain Rule

$$
 y=x^2,\qquad z=\sin(y)
$$

$$
\frac{dz}{dx}=\frac{dz}{dy}\frac{dy}{dx}=\cos(x^2)\,(2x)
$$

In [3]:
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

z.backward()
x.grad

tensor(-7.6613)

In [4]:
with torch.no_grad():
    manual_gradient = 2 * x * torch.cos(x ** 2)

manual_gradient

tensor(-7.6613)

## 3. Vector Gradients

`mean()` gives one scalar loss for all elements.

$$
L=\frac{x_1^2+x_2^2+x_3^2}{3},\qquad \frac{\partial L}{\partial x_i}=\frac{2x_i}{3}
$$

In [5]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
loss = (x ** 2).mean()

loss

tensor(4.6667, grad_fn=<MeanBackward0>)

In [6]:
loss.backward()
x.grad

tensor([0.6667, 1.3333, 2.0000])

## 4. Gradient Accumulation

Each `backward()` **adds** to `.grad`. Clear it before the next independent step.

In [7]:
w = torch.tensor(2.0, requires_grad=True)
(w ** 2).backward()
w.grad

tensor(4.)

In [8]:
(w ** 2).backward()
w.grad

tensor(8.)

In [9]:
w.grad.zero_()
w.grad

tensor(0.)

## 5. `torch.no_grad()`

Inside `torch.no_grad()`, new operations are not recorded for backpropagation.

In [10]:
x = torch.tensor(2.0, requires_grad=True)
tracked = x ** 2

with torch.no_grad():
    not_tracked = x ** 2

tracked.requires_grad, not_tracked.requires_grad

(True, False)

## 6. Manual Gradient Descent

$$
 w_{\mathrm{new}}=w-\eta\frac{\partial L}{\partial w}
$$

$$
 b_{\mathrm{new}}=b-\eta\frac{\partial L}{\partial b}
$$

Here, $\eta$ is the learning rate. Use `torch.no_grad()` to update parameters.

In [11]:
x = torch.tensor(2.0)
y = torch.tensor(5.0)
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

prediction = w * x + b
loss = (prediction - y) ** 2
loss.backward()

w.grad, b.grad

(tensor(-20.), tensor(-10.))

In [12]:
learning_rate = 0.01

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

w.grad.zero_()
b.grad.zero_()

w, b

(tensor(0.2000, requires_grad=True), tensor(0.1000, requires_grad=True))

## 7. Tensor and Matrix Shapes

Rows = samples. Columns = features.

$$
X\in\mathbb{R}^{m\times d},\qquad W\in\mathbb{R}^{d\times 1}
$$

$$
Z=XW+b\in\mathbb{R}^{m\times 1}
$$

In [13]:
X = torch.tensor([[1.], [2.], [3.], [4.], [5.]])
W = torch.tensor([[2.]])
b = torch.tensor([1.])

X.shape, W.shape, (X @ W + b).shape

(torch.Size([5, 1]), torch.Size([1, 1]), torch.Size([5, 1]))

In [14]:
X_multi = torch.tensor([
    [1., 2., 3.],
    [2., 3., 4.],
    [3., 4., 5.]
])
W_multi = torch.tensor([[1.], [0.5], [-1.]])
b_multi = torch.zeros(1)

X_multi.shape, W_multi.shape, (X_multi @ W_multi + b_multi).shape

(torch.Size([3, 3]), torch.Size([3, 1]), torch.Size([3, 1]))

## 8. Manual MSE Loss

Mean squared error for regression:

$$
\boxed{\mathrm{MSE}=\frac{1}{m}\sum_{i=1}^{m}(\hat y_i-y_i)^2}
$$

In [15]:
def mse_loss(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean()

mse_loss(torch.tensor([2., 4.]), torch.tensor([3., 5.]))

tensor(1.)

## 9. Manual Binary Cross-Entropy

For binary labels $y_i\in\{0,1\}$ and predicted probabilities $p_i$:

$$
\boxed{\mathrm{BCE}=-\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]}
$$

`clamp()` avoids $\log(0)$ in this educational implementation.

In [16]:
def binary_cross_entropy_loss(probabilities, y_true):
    eps = 1e-7
    p = torch.clamp(probabilities, min=eps, max=1 - eps)
    return -(y_true * torch.log(p) +
             (1 - y_true) * torch.log(1 - p)).mean()

binary_cross_entropy_loss(
    torch.tensor([[0.8], [0.3]]),
    torch.tensor([[1.], [0.]])
)

tensor(0.2899)

## 10. Check Autograd Against Manual Derivatives

For one regression sample:

$$
\hat y=wx+b,\qquad L=(\hat y-y)^2
$$

$$
\frac{\partial L}{\partial w}=2(\hat y-y)x,\qquad
\frac{\partial L}{\partial b}=2(\hat y-y)
$$

In [17]:
x = torch.tensor(2.0)
y = torch.tensor(5.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

y_pred = w * x + b
loss = mse_loss(y_pred, y)
loss.backward()

with torch.no_grad():
    dw_manual = 2 * (y_pred - y) * x
    db_manual = 2 * (y_pred - y)

(w.grad, dw_manual), (b.grad, db_manual)

((tensor(-12.), tensor(-12.)), (tensor(-6.), tensor(-6.)))

## 11. Linear Regression — Full Training

Five samples, one feature. The target is $y=2x+1$.

$$
\hat Y=XW+b
$$

$$
L=\frac{1}{m}\sum_{i=1}^{m}(\hat y_i-y_i)^2
$$

In [18]:
X_lr = torch.tensor([[1.], [2.], [3.], [4.], [5.]])
y_lr = torch.tensor([[3.], [5.], [7.], [9.], [11.]])

X_lr.shape, y_lr.shape

(torch.Size([5, 1]), torch.Size([5, 1]))

In [19]:
W_lr = torch.zeros((1, 1), requires_grad=True)
b_lr = torch.zeros(1, requires_grad=True)

learning_rate = 0.01
epochs = 1000
linear_losses = []

for epoch in range(epochs):
    y_pred = X_lr @ W_lr + b_lr
    loss = mse_loss(y_pred, y_lr)

    loss.backward()

    with torch.no_grad():
        W_lr -= learning_rate * W_lr.grad
        b_lr -= learning_rate * b_lr.grad

    W_lr.grad.zero_()
    b_lr.grad.zero_()

    linear_losses.append(loss.item())

In [20]:
linear_losses[::200] + [linear_losses[-1]]

[57.0,
 0.008059590123593807,
 0.0020796258468180895,
 0.0005366100231185555,
 0.00013846071669831872,
 3.5972188925370574e-05]

In [21]:
W_lr, b_lr

(tensor([[2.0039]], requires_grad=True), tensor([0.9860], requires_grad=True))

In [22]:
with torch.no_grad():
    new_x = torch.tensor([[6.]])
    new_y = new_x @ W_lr + b_lr

new_y

tensor([[13.0093]])

## 12. Logistic Regression — Full Training

Binary classification uses sigmoid and binary cross-entropy.

$$
Z=XW+b
$$

$$
 p=\sigma(Z)=\frac{1}{1+e^{-Z}}
$$

$$
\hat y_{\mathrm{class}}=\begin{cases}1,&p\geq0.5\\0,&p<0.5\end{cases}
$$

In [23]:
X_log = torch.tensor([[1.], [2.], [3.], [4.], [5.], [6.]])
y_log = torch.tensor([[0.], [0.], [0.], [1.], [1.], [1.]])

X_log.shape, y_log.shape

(torch.Size([6, 1]), torch.Size([6, 1]))

In [24]:
W_log = torch.zeros((1, 1), requires_grad=True)
b_log = torch.zeros(1, requires_grad=True)

learning_rate = 0.1
epochs = 1000
logistic_losses = []

for epoch in range(epochs):
    logits = X_log @ W_log + b_log
    probabilities = torch.sigmoid(logits)
    loss = binary_cross_entropy_loss(probabilities, y_log)

    loss.backward()

    with torch.no_grad():
        W_log -= learning_rate * W_log.grad
        b_log -= learning_rate * b_log.grad

    W_log.grad.zero_()
    b_log.grad.zero_()

    logistic_losses.append(loss.item())

In [25]:
logistic_losses[::200] + [logistic_losses[-1]]

[0.6931471824645996,
 0.32523712515830994,
 0.23696863651275635,
 0.19476318359375,
 0.16921214759349823,
 0.1517137885093689]

In [26]:
W_log, b_log

(tensor([[1.7036]], requires_grad=True), tensor([-5.6903], requires_grad=True))

In [27]:
with torch.no_grad():
    probabilities = torch.sigmoid(X_log @ W_log + b_log)
    classes = (probabilities >= 0.5).int()
    accuracy = (classes == y_log).float().mean()

probabilities, classes, accuracy

(tensor([[0.0182],
         [0.0925],
         [0.3591],
         [0.7548],
         [0.9442],
         [0.9893]]),
 tensor([[0],
         [0],
         [0],
         [1],
         [1],
         [1]], dtype=torch.int32),
 tensor(1.))

## 13. Logistic Regression Gradient Check

For one sample using sigmoid with BCE:

$$
\frac{\partial L}{\partial w}=(p-y)x,\qquad
\frac{\partial L}{\partial b}=p-y
$$

In [28]:
x = torch.tensor(2.0)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

p = torch.sigmoid(w * x + b)
loss = binary_cross_entropy_loss(p, y)
loss.backward()

with torch.no_grad():
    dw_manual = (p - y) * x
    db_manual = p - y

(w.grad, dw_manual), (b.grad, db_manual)

((tensor(1.7616), tensor(1.7616)), (tensor(0.8808), tensor(0.8808)))

## 14. Interview Questions — Autograd and Regression

**1. What is Autograd?** Automatic differentiation that tracks operations to calculate gradients.

**2. What does `requires_grad=True` do?** Enables gradient tracking for operations involving the tensor.

**3. What does `backward()` do?** Computes gradients by applying the chain rule from the scalar output backward.

**4. Why must the output usually be scalar for `backward()` without arguments?** Non-scalar outputs need an explicit upstream gradient (or a reduction like `mean()`).

**5. Why does `.grad` accumulate?** Multiple backward calls add contributions rather than replacing them.

**6. Why call `zero_()`?** To remove gradients from the previous training step.

**7. What does `torch.no_grad()` do?** Prevents new operations from being recorded in the autograd graph.

**8. Why update weights inside `torch.no_grad()`?** So the update does not create a new gradient-tracked operation or modify trainable leaf tensors while tracking.

**9. What is a leaf tensor?** A tensor created by the user (not as a result of a tracked operation), often holding trainable parameters.

**10. What is the difference between `*` and `@`?** `*` is element-wise multiplication; `@` is matrix multiplication (or vector dot product).

**11. What does shape `(5, 3)` mean?** Five samples with three features each.

**12. What is gradient descent?** Updating parameters in the negative-gradient direction: $w\leftarrow w-\eta\,\partial L/\partial w$.

**13. Why MSE for linear regression?** It penalizes squared prediction errors for continuous targets.

**14. What makes logistic regression a classification model?** Sigmoid maps logits to probabilities, then a threshold converts them to classes.

**15. Why use BCE for binary classification?** It penalizes predicted probabilities according to the correct class.

**16. What is a logit?** The raw model output $XW+b$ before the sigmoid.

**17. Why do we use `clamp()` in manual BCE?** To avoid undefined $\log(0)$.

**18. What is the benefit of vectorization?** Matrix operations process many samples simultaneously and avoid Python loops over samples.

**19. Does `torch.no_grad()` erase existing gradients?** No. It only changes whether new operations are recorded.

**20. What is the practical production equivalent?** `nn.Linear` with `nn.MSELoss` for regression; `nn.Linear` with `nn.BCEWithLogitsLoss` for binary classification, typically using an optimizer.